## Parte 1: Carregamento e visão geral da base

In [1]:
# Visão geral
import pandas as pd
df = pd.read_csv("Base Varejo.csv", sep=";")
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 830000 entries, 0 to 829999
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   DATA         830000 non-null  str    
 1   CO_ID        830000 non-null  int64  
 2   CL_ID        830000 non-null  int64  
 3   CL_GENERO    830000 non-null  str    
 4   CL_EC        830000 non-null  int64  
 5   CL_FHL       830000 non-null  int64  
 6   CL_SEG       830000 non-null  str    
 7   PR_ID        830000 non-null  int64  
 8   PR_CAT       830000 non-null  str    
 9   PR_NOME      830000 non-null  str    
 10  Unnamed: 10  0 non-null       float64
 11  Unnamed: 11  0 non-null       float64
 12  Unnamed: 12  0 non-null       float64
 13  Unnamed: 13  0 non-null       float64
dtypes: float64(4), int64(5), str(5)
memory usage: 88.7 MB


## Parte 2: Diagnóstico de problemas

In [2]:
# Conta os valores vazios em cada coluna
print("Nulos por coluna:")
print(df.isnull().sum())

# Linhas duplicadas
print("\nLinhas duplicadas:", df.duplicated().sum())

# Categorias de produto
print("\nCategorias de produto:")
print(df["PR_CAT"].value_counts().head(10))

#Produtos: quantos nomes diferentes existem
print("\nProdutos distintos:", df["PR_NOME"].nunique())

# Datas, ver exemplos e testar se alguma é inválida
datas_convertidas = pd.to_datetime(df["DATA"], format="%d/%m/%Y")
print("\nPrimeira data:", datas_convertidas.min())
print("Última data:", datas_convertidas.max())


Nulos por coluna:
DATA                0
CO_ID               0
CL_ID               0
CL_GENERO           0
CL_EC               0
CL_FHL              0
CL_SEG              0
PR_ID               0
PR_CAT              0
PR_NOME             0
Unnamed: 10    830000
Unnamed: 11    830000
Unnamed: 12    830000
Unnamed: 13    830000
dtype: int64

Linhas duplicadas: 96553

Categorias de produto:
PR_CAT
ALIMENTOS     434767
HIGIENE       155574
LIMPEZA       145754
BEBIDAS        43299
PET            32399
ACESSORIOS     14557
#N/D            3650
Name: count, dtype: int64

Produtos distintos: 118

Primeira data: 2019-01-04 00:00:00
Última data: 2022-12-08 00:00:00


**Problemas encontrados:**
1. 4 colunas 10 a 13 (Unnamed) totalmente vazias
2. A coluna DATA está como texto, e não como data
3. Duplicadas: 96553
4. PR_CAT vendas "#N/D": 3.650 (categoria vazia disfarçada de texto)

## Parte 3:	Limpeza dos dados

Aqui vou arrumar os problemas que achei na parte 2:
- apagar as 4 colunas vazias
- trocar o #N/D das categorias por "Sem Categoria"
- tirar as linhas repetidas
- mudar a coluna DATA de texto para data

In [ ]:
# apagar as 4 colunas vazias
df = df.drop(columns=["Unnamed: 10", "Unnamed: 11", "Unnamed: 12", "Unnamed: 13"])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 830000 entries, 0 to 829999
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   DATA       830000 non-null  str  
 1   CO_ID      830000 non-null  int64
 2   CL_ID      830000 non-null  int64
 3   CL_GENERO  830000 non-null  str  
 4   CL_EC      830000 non-null  int64
 5   CL_FHL     830000 non-null  int64
 6   CL_SEG     830000 non-null  str  
 7   PR_ID      830000 non-null  int64
 8   PR_CAT     830000 non-null  str  
 9   PR_NOME    830000 non-null  str  
dtypes: int64(5), str(5)
memory usage: 63.3 MB


In [4]:
# trocar o #N/D das categorias por "Sem Categoria"
def arrumar_categoria(categoria):
    if categoria == "#N/D":
        return "Sem Categoria"
    else:
        return categoria
df["PR_CAT"] = df["PR_CAT"].apply(arrumar_categoria)
print(df["PR_CAT"].value_counts().head(10))

PR_CAT
ALIMENTOS        434767
HIGIENE          155574
LIMPEZA          145754
BEBIDAS           43299
PET               32399
ACESSORIOS        14557
Sem Categoria      3650
Name: count, dtype: int64


Escolhi não apagar as linhas com #N/D porque são vendas de verdade, só faltava a categoria. Se apagasse, perderia 3.650 vendas. Troquei por “Sem Categoria” para não perder os dados e deixar o problema visível.

In [5]:
# apagar as linhas repetidas
print("\nLinhas duplicadas:", df.duplicated().sum())
df = df.drop_duplicates()
print("\nLinhas após remoção de duplicadas:", len(df))



Linhas duplicadas: 96553

Linhas após remoção de duplicadas: 733447


In [6]:
# mudar a coluna DATA de texto para data
df["DATA"] = pd.to_datetime(df["DATA"], format='%d/%m/%Y')
df.info()



<class 'pandas.DataFrame'>
Index: 733447 entries, 0 to 829999
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   DATA       733447 non-null  datetime64[us]
 1   CO_ID      733447 non-null  int64         
 2   CL_ID      733447 non-null  int64         
 3   CL_GENERO  733447 non-null  str           
 4   CL_EC      733447 non-null  int64         
 5   CL_FHL     733447 non-null  int64         
 6   CL_SEG     733447 non-null  str           
 7   PR_ID      733447 non-null  int64         
 8   PR_CAT     733447 non-null  str           
 9   PR_NOME    733447 non-null  str           
dtypes: datetime64[us](1), int64(5), str(4)
memory usage: 61.6 MB


In [7]:
# ver quantas compras diferentes tem (CO_ID é o numero da compra)
print("\nCompras distintas:", df["CO_ID"].nunique())
print("\ntotal de itens:",len(df))




Compras distintas: 18471

total de itens: 733447


cada compra tem, em média 40 itens (733447/18471=40).

In [ ]:
# salvar a base tratada em um novo arquivo CSV
df.to_csv("df_limpo.csv", sep=";", index=False)


## Parte 4: Estatísticas Descritiva de “nº de filhos clientes” (CL_FHL)

In [10]:
# deixar so 1 linha por cliente, pra nao contar o mesmo cliente varias vezes
df_clientes = df.drop_duplicates(subset=["CL_ID"])

# estatísticas do nº filhos por cliente
print("media:", df_clientes["CL_FHL"].mean())
print("mediana:", df_clientes["CL_FHL"].median())
print("Desvio padrão:", df_clientes["CL_FHL"].std())
print("Moda:", df_clientes["CL_FHL"].mode()[0])
print("Min:", df_clientes["CL_FHL"].min())
print("Max:", df_clientes["CL_FHL"].max())
print("Contagem:", df_clientes["CL_FHL"].count())


media: 1.136
mediana: 0.0
Desvio padrão: 1.4133299337032035
Moda: 0
Min: 0
Max: 4
Contagem: 1000


Olhando os 1000 clientes, a maioria não tem filhos (a mediana e a moda deram 0). A média deu 1 filho por cliente, mas ela fica um pouco mais alta porque alguns clientes têm 3 ou 4 filhos. O mínimo é 0 e o máximo é 4 filhos. O desvio padrão de 1,4 mostra que as famílias variam bastante, tem muita gente sem filhos.

## Parte 5: Agrupamentos (CL_GENERO; PR_CAT)
Explorar padrões de agrupamento (gênero com mais vendas, compras), usando groupby() ou pivot_table().

In [12]:
# quantidade de itens comprados por genero
print(df.groupby("CL_GENERO")["PR_ID"].count())

# itens comprados por categoria e genero
tabela_categoria_genero = pd.pivot_table(df, index=["PR_CAT"], columns="CL_GENERO", values="PR_ID", aggfunc="count")
print(tabela_categoria_genero)



CL_GENERO
F    382427
M    351020
Name: PR_ID, dtype: int64
CL_GENERO           F       M
PR_CAT                       
ACESSORIOS       6839    6032
ALIMENTOS      200274  183923
BEBIDAS         19764   18500
HIGIENE         71721   65981
LIMPEZA         67328   61304
PET             14809   13744
Sem Categoria    1692    1536


As mulheres compram mais, cerca de 31 mil itens a mais. Ranking das categoria: Alimentos; Higiene; Limpeza; Bebidas; Pet e Acessórios. Alimentos sozinho é mais da metade das vendas. Porém, o perfil de compra entre os gêneros não tem nenhuma discrepância.


## Parte 6: Conclusões